# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [58]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Chart Type: Indexed Line Chart (px.line)Justification: A line chart is the standard encoding for continuous temporal trajectories, allowing direct comparison of growth rates via line slopes across decades. Indexing each continent's population to a common baseline ($1970 = 100$) removes initial population size differences so relative growth rates can be compared fairly on a single axis.

In [59]:
# Q1
# YOUR CODE HERE
import numpy as np
import pandas as pd
import plotly.express as px

cont_pop = (
    df_long.groupby(["Continent", "Year"], as_index=False)["Population"]
    .sum()
    .sort_values(["Continent", "Year"])
)

pop_1970 = cont_pop[cont_pop["Year"] == 1970].set_index("Continent")["Population"]
cont_pop["Index_1970"] = (
    cont_pop["Population"] / cont_pop["Continent"].map(pop_1970)
) * 100

highlight = "Africa"
color_map = {
    c: ("#0072B2" if c == highlight else "#B0B0B0")  
    for c in cont_pop["Continent"].unique()
}

fig = px.line(
    cont_pop,
    x="Year",
    y="Index_1970",
    color="Continent",
    color_discrete_map=color_map,
    template="plotly_white",
    title=(
        "<b>Africa's population nearly quadrupled since 1970, outpacing every other continent</b>"
        "<br><span style='font-size:13px; color:#555555;'>"
        "Total continental population indexed to 1970 (1970 = 100), 1970–2022</span>"
    ),
)

for trace in fig.data:
  is_africa = trace.name == highlight
  trace.update(
      line=dict(width=3.5 if is_africa else 1.8),
      opacity=1.0 if is_africa else 0.75,
  )

fig.data = tuple(sorted(fig.data, key=lambda t: t.name == highlight))

latest = cont_pop[cont_pop["Year"] == 2022].copy()

y_offsets = {
    "Africa": 0,
    "Asia": 2,
    "Oceania": -2,
    "South America": 2,
    "North America": -2,
    "Europe": 0,
}

for _, row in latest.iterrows():
  c = row["Continent"]
  val = row["Index_1970"]
  is_africa = c == highlight
  fig.add_annotation(
      x=2022.4,
      y=val + y_offsets.get(c, 0),
      text=f"<b>{c} ({val:.0f})</b>" if is_africa else f"{c} ({val:.0f})",
      showarrow=False,
      xanchor="left",
      yanchor="middle",
      font=dict(
          size=12 if is_africa else 11,
          color="#0072B2" if is_africa else "#555555",
      ),
  )

fig.add_hline(y=100, line_dash="dot", line_color="#CCCCCC", line_width=1)

fig.update_layout(
    showlegend=False,
    margin=dict(l=50, r=145, t=80, b=45),
    hovermode="x unified",
)

fig.update_xaxes(
    title=None,
    showgrid=False,
    tickvals=[1970, 1980, 1990, 2000, 2010, 2022],
    range=[1969, 2023],
    linecolor="#CCCCCC",
)

fig.update_yaxes(
    title="Population Index (1970 = 100)",
    showgrid=True,
    gridcolor="#EEEEEE",
    zeroline=False,
    range=[85, 420],
)

fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

Chart Type: Sorted Horizontal Bar Chart (px.bar)

Justification: A sorted bar chart makes ranking and comparing exact magnitude differences across 15 discrete countries effortless via aligned bar lengths along a common baseline. Colouring African countries in Okabe-Ito blue (#0072B2) against neutral grey (#B0B0B0) for other regions directly answers how many top-gaining countries are in Africa without needing a legend, while direct end-of-bar value labels and a callout annotation highlight the key takeaways.

In [60]:
# Q2
# YOUR CODE HERE

import pandas as pd
import plotly.express as px

pop_wide = df[["CCA3", "Country", "Continent", "1970 Population", "2022 Population"]].copy()
pop_wide["Growth_1970_2022"] = pop_wide["2022 Population"] - pop_wide["1970 Population"]
pop_wide["Growth_Millions"] = pop_wide["Growth_1970_2022"] / 1e6

top15 = (
    pop_wide.sort_values("Growth_1970_2022", ascending=False)
    .head(15)
    .sort_values("Growth_1970_2022", ascending=True)
    .copy()
)

top15["Is_Africa"] = top15["Continent"] == "Africa"
africa_count = int(top15["Is_Africa"].sum())

color_map = {
    True: "#0072B2",   # Accent blue for Africa
    False: "#B0B0B0",  # Neutral grey for other continents
}

fig = px.bar(
    top15,
    x="Growth_Millions",
    y="Country",
    orientation="h",
    color="Is_Africa",
    color_discrete_map=color_map,
    template="plotly_white",
    text=top15["Growth_Millions"].map(lambda v: f"+{v:.1f}M"),
    title=(
        f"<b>India and China lead absolute growth, while {africa_count} of the top 15 are in Africa</b>"
        "<br><span style='font-size:13px; color:#555555;'>"
        "Absolute population added (millions), 1970–2022; blue highlights African countries</span>"
    ),
)

fig.update_traces(
    textposition="outside",
    cliponaxis=False,
    marker_line_width=0,
    hovertemplate="<b>%{y}</b><br>Added: %{x:.1f}M<extra></extra>",
)

fig.add_annotation(
    xref="paper",
    yref="paper",
    x=0.98,
    y=0.14,
    text=f"<b>{africa_count} of 15</b> in Africa<br><span style='font-size:11px; color:#555555;'>(Nigeria, DR Congo, Ethiopia)</span>",
    showarrow=False,
    align="right",
    bgcolor="#F7F9FB",
    bordercolor="#0072B2",
    borderwidth=1,
    borderpad=6,
    font=dict(size=12, color="#0072B2"),
)

fig.update_layout(
    showlegend=False,
    margin=dict(l=120, r=85, t=80, b=35),
    hovermode="closest",
)

fig.update_xaxes(
    title=None,
    showticklabels=False,
    showgrid=False,
    zeroline=False,
)

fig.update_yaxes(
    title=None,
    showgrid=False,
    linecolor="#CCCCCC",
)

fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
Chart Type: Scatter Plot (px.scatter with log-log axes)

Justification: A scatter plot is ideal for revealing correlation, spread, and outliers between two continuous variables (land area and 2022 population) across 234 countries. Using log scales on both axes prevents huge countries like Russia, Canada, and China from squishing smaller nations into a corner, while colouring one highlighted continent in Okabe-Ito blue (#0072B2) against neutral grey (#B0B0B0) context points and directly labelling notable outliers makes exceptions immediately readable.

In [61]:
# Q3
# YOUR CODE HERE
import numpy as np
import pandas as pd
import plotly.express as px

pop_area = df[
    (df["Area"] > 0) & (df["2022 Population"] > 0)
].copy()
pop_area["Pop_Millions"] = pop_area["2022 Population"] / 1e6

highlight_continent = "Asia"
color_map = {
    c: ("#0072B2" if c == highlight_continent else "#B0B0B0")
    for c in pop_area["Continent"].unique()
}

fig = px.scatter(
    pop_area,
    x="Area",
    y="Pop_Millions",
    color="Continent",
    color_discrete_map=color_map,
    log_x=True,
    log_y=True,
    template="plotly_white",
    hover_name="Country",
    hover_data={
        "Continent": True,
        "Area": ":,.0f",
        "Pop_Millions": ":.2f",
    },
    title=(
        "<b>Land area alone barely predicts population—dense outliers break the trend</b>"
        "<br><span style='font-size:13px; color:#555555;'>"
        "Land area (km²) vs. 2022 population (millions), log-log scale; Asia highlighted in blue</span>"
    ),
)

for trace in fig.data:
  is_asia = trace.name == highlight_continent
  trace.update(
      marker=dict(
          size=8 if is_asia else 6.5,
          opacity=0.85 if is_asia else 0.55,
          line=dict(width=0.8, color="#FFFFFF" if is_asia else "#888888"),
      )
  )

fig.data = tuple(sorted(fig.data, key=lambda t: t.name == highlight_continent))

outliers = [
    ("India", "top right", "#0072B2"),
    ("China", "top left", "#0072B2"),
    ("Bangladesh", "top left", "#0072B2"),
    ("Russia", "bottom right", "#444444"),
    ("Canada", "bottom right", "#444444"),
    ("Australia", "bottom right", "#444444"),
    ("Greenland", "bottom left", "#444444"),
]

lookup = pop_area.set_index("Country")

for country, pos, txt_color in outliers:
  if country in lookup.index:
    row = lookup.loc[country]
    if isinstance(row, pd.DataFrame):
      row = row.iloc[0]
    x_val = float(row["Area"])
    y_val = float(row["Pop_Millions"])

    dx, dy, xanchor, yanchor = 0, 0, "left", "bottom"
    if pos == "top right":
      dx, dy, xanchor, yanchor = 1.08, 1.08, "left", "bottom"
    elif pos == "top left":
      dx, dy, xanchor, yanchor = 0.92, 1.12, "right", "bottom"
    elif pos == "bottom right":
      dx, dy, xanchor, yanchor = 1.08, 0.88, "left", "top"
    elif pos == "bottom left":
      dx, dy, xanchor, yanchor = 0.95, 0.85, "right", "top"

    fig.add_annotation(
        x=x_val * dx,
        y=y_val * dy,
        text=f"<b>{country}</b>",
        showarrow=True,
        arrowhead=0,
        arrowwidth=1,
        arrowcolor="#999999",
        ax=0,
        ay=0,
        xanchor=xanchor,
        yanchor=yanchor,
        font=dict(size=11, color=txt_color),
    )

fig.update_layout(
    showlegend=False,
    margin=dict(l=55, r=40, t=80, b=45),
    hovermode="closest",
)

fig.update_xaxes(
    title=None,
    showgrid=False,
    linecolor="#CCCCCC",
    tickvals=[1e3, 1e4, 1e5, 1e6, 1e7],
    ticktext=["1k km²", "10k km²", "100k km²", "1M km²", "10M km²"],
)

fig.update_yaxes(
    title=None,
    showgrid=True,
    gridcolor="#EEEEEE",
    zeroline=False,
    tickvals=[0.01, 0.1, 1, 10, 100, 1000],
    ticktext=["0.01M", "0.1M", "1M", "10M", "100M", "1B"],
)

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

Chart Type: Box Plot (px.box with log_y=True and data points overlaid)

Justification: A box plot shows the median, interquartile range (spread/variability), whiskers, and extreme outliers within each continent side by side, making within- and cross-continent density spread easy to compare. Because population density (People per km²) is heavily right-skewed across countries and territories, a logarithmic y-axis keeps both typical values and extreme outliers legible without compressing the boxes.

In [62]:
# Q4
# YOUR CODE HERE

import pandas as pd
import plotly.express as px


dens_df = df[df["Density"] > 0].copy()

continent_order = (
    dens_df.groupby("Continent")["Density"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

highlight_continent = "Asia"
color_map = {
    c: ("#0072B2" if c == highlight_continent else "#B0B0B0")
    for c in continent_order
}

fig = px.box(
    dens_df,
    x="Continent",
    y="Density",
    color="Continent",
    category_orders={"Continent": continent_order},
    color_discrete_map=color_map,
    log_y=True,
    points="all",
    template="plotly_white",
    hover_name="Country",
    hover_data={"Continent": False, "Density": ":,.1f"},
    title=(
        "<b>Asia has the highest median density, while wide spreads span every continent</b>"
        "<br><span style='font-size:13px; color:#555555;'>"
        "Population density (people per km², log scale) across countries/territories, 2022</span>"
    ),
)

for trace in fig.data:
  is_asia = trace.name == highlight_continent
  fill_rgba = "rgba(0, 114, 178, 0.22)" if is_asia else "rgba(176, 176, 176, 0.22)"
  stroke_hex = "#0072B2" if is_asia else "#6E6E6E"
  trace.update(
      fillcolor=fill_rgba,
      line=dict(color=stroke_hex, width=1.8),
      marker=dict(
          color=stroke_hex,
          size=4.5,
          opacity=0.55,
      ),
      jitter=0.35,
      pointpos=0,
  )

asia_median = dens_df[dens_df["Continent"] == highlight_continent]["Density"].median()
fig.add_annotation(
    x=highlight_continent,
    y=asia_median,
    text=f"Median: {asia_median:.0f}/km²",
    showarrow=True,
    arrowhead=0,
    arrowwidth=1,
    arrowcolor="#0072B2",
    ax=58,
    ay=-14,
    font=dict(size=11, color="#0072B2"),
    bgcolor="#FFFFFF",
)

fig.update_layout(
    showlegend=False,
    margin=dict(l=55, r=30, t=80, b=45),
    hovermode="closest",
)

fig.update_xaxes(
    title=None,
    showgrid=False,
    linecolor="#CCCCCC",
)

fig.update_yaxes(
    title=None,
    showgrid=True,
    gridcolor="#EEEEEE",
    zeroline=False,
    tickvals=[0.1, 1, 10, 100, 1000, 10000],
    ticktext=["0.1", "1", "10", "100", "1k", "10k"],
)

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

Chart Type: Choropleth Map (px.choropleth with locationmode='ISO-3')

Justification: A choropleth map directly answers "where in the world" by encoding country-level percentage population change onto geographic boundaries using CCA3 ISO-3 codes. Using a CVD-safe diverging colour scale (PuOr — orange for population decline, light neutral grey at 0% midpoint, and purple for population growth) separates shrinking nations from growing ones immediately without relying on red/green.

In [63]:
# Q5
# YOUR CODE HERE

import pandas as pd
import plotly.express as px

pop_change = df[["CCA3", "Country", "Continent", "2000 Population", "2022 Population"]].copy()
pop_change["Pct_Change_2000_2022"] = (
    (pop_change["2022 Population"] - pop_change["2000 Population"])
    / pop_change["2000 Population"]
) * 100

shrinking_count = int((pop_change["Pct_Change_2000_2022"] < 0).sum())

fig = px.choropleth(
    pop_change,
    locations="CCA3",
    locationmode="ISO-3",
    color="Pct_Change_2000_2022",
    hover_name="Country",
    hover_data={
        "CCA3": False,
        "Continent": True,
        "Pct_Change_2000_2022": ":.1f",
    },
    color_continuous_scale="PuOr",
    color_continuous_midpoint=0,
    range_color=[-30, 100],
    template="plotly_white",
    title=(
        "<b>Population shrank across Eastern Europe and Japan while surging in Africa and the Middle East</b>"
        "<br><span style='font-size:13px; color:#555555;'>"
        f"Percentage population change, 2000–2022 ({shrinking_count} countries/territories declined; scale clipped at -30% and +100%)</span>"
    ),
)


fig.update_traces(
    marker_line_color="#FFFFFF",
    marker_line_width=0.5,
    hovertemplate=(
        "<b>%{hovertext}</b> (%{customdata[0]})<br>"
        "Change (2000–2022): <b>%{z:+.1f}%</b><extra></extra>"
    ),
)

fig.update_geos(
    projection_type="natural earth",
    showframe=False,
    showcoastlines=False,
    showland=True,
    landcolor="#F2F2F2",
    showocean=False,
    lataxis_range=[-58, 84],
)

fig.update_layout(
    margin=dict(l=10, r=10, t=80, b=20),
    coloraxis_colorbar=dict(
        title="Change (%)",
        ticksuffix="%",
        tickvals=[-30, 0, 50, 100],
        ticktext=["≤ -30%", "0%", "+50%", "≥ +100%"],
        len=0.55,
        thickness=12,
        y=0.5,
    ),
)

fig.add_annotation(
    xref="paper",
    yref="paper",
    x=0.63,
    y=0.76,
    text="<b>Eastern Europe & Baltics</b><br><span style='font-size:11px;'>Sharpest declines (e.g., Ukraine, Latvia, Lithuania, Bulgaria)</span>",
    showarrow=True,
    arrowhead=0,
    arrowwidth=1,
    arrowcolor="#B35806",
    ax=65,
    ay=-32,
    align="left",
    bgcolor="#FFFFFF",
    bordercolor="#B35806",
    borderwidth=1,
    borderpad=4,
    font=dict(size=11, color="#7F3B08"),
)

fig.add_annotation(
    xref="paper",
    yref="paper",
    x=0.55,
    y=0.45,
    text="<b>Sub-Saharan Africa & Gulf</b><br><span style='font-size:11px;'>Widespread growth (+50% to +100%+)</span>",
    showarrow=True,
    arrowhead=0,
    arrowwidth=1,
    arrowcolor="#542788",
    ax=-75,
    ay=42,
    align="left",
    bgcolor="#FFFFFF",
    bordercolor="#542788",
    borderwidth=1,
    borderpad=4,
    font=dict(size=11, color="#542788"),
)

fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

Chart Type: Treemap (px.treemap)Justification: A treemap encodes hierarchical part-to-whole data (Continent → Country) as nested rectangles whose areas are directly proportional to 2022 population, displaying both continental totals and individual country shares on a single screen. Highlighting Asia in Okabe-Ito blue (#0072B2) against neutral grey tones (#D0D0D0) for the remaining continents emphasizes that nearly 60% of humanity lives in Asia—anchored by China and India—while white borders, hidden pathbars, and uniformtext keep tiny territories readable without clutter.   

In [64]:
# Q6
# YOUR CODE HERE

import pandas as pd
import plotly.express as px

pop_2022 = df[["Country", "Continent", "2022 Population", "World share (%)"]].copy()
asia_share = pop_2022.loc[pop_2022["Continent"] == "Asia", "World share (%)"].sum()

color_map = {
    "Asia": "#0072B2",
    "Africa": "#CFCFCF",
    "Europe": "#D6D6D6",
    "North America": "#DCDCDC",
    "South America": "#E2E2E2",
    "Oceania": "#E8E8E8",
}

fig = px.treemap(
    pop_2022,
    path=["Continent", "Country"],
    values="2022 Population",
    color="Continent",
    color_discrete_map=color_map,
    template="plotly_white",
    title=(
        f"<b>Nearly 60% of the world lives in Asia—led by China and India</b>"
        "<br><span style='font-size:13px; color:#555555;'>"
        f"2022 global population (~8.0B) by continent and country (Asia = {asia_share:.1f}% of world total)</span>"
    ),
)

fig.update_traces(
    root_color="#FFFFFF",
    marker=dict(
        line=dict(width=1.2, color="#FFFFFF"),
        pad=dict(t=22, l=3, r=3, b=3),
    ),
    texttemplate="<b>%{label}</b><br>%{percentRoot:.1%}",
    textposition="middle center",
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Population: %{value:,.0f}<br>"
        "Share of world: %{percentRoot:.2%}<extra></extra>"
    ),
)


fig.update_layout(
    uniformtext=dict(minsize=10, mode="hide"),
    margin=dict(l=10, r=10, t=80, b=15),
)

fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

Chart Type: 100% Stacked Bar Chart (px.bar with barnorm='percent')

Justification: A 100% stacked bar chart shows part-to-whole composition across discrete census years (1970 to 2022) where every column sums to 100%, making shifts in each group's share of the global population immediately readable. Highlighting Africa in Okabe-Ito blue (#0072B2) against muted context shades for Asia, Europe, and Rest of the world draws attention to Africa's steady rise while keeping the full composition clear without cluttering the chart with a legend.

In [65]:
import pandas as pd
import plotly.express as px

# 1. Map continents into Africa, Asia, Europe, and Rest of the world
focus_continents = ["Africa", "Asia", "Europe"]
share_df = df_long.copy()
share_df["Group"] = share_df["Continent"].where(
    share_df["Continent"].isin(focus_continents),
    other="Rest of the world",
)

# 2. Aggregate population by Year and Group, then compute % share of each year's world total
grp_pop = (
    share_df.groupby(["Year", "Group"], as_index=False)["Population"]
    .sum()
    .sort_values(["Year", "Group"])
)
year_totals = grp_pop.groupby("Year")["Population"].transform("sum")
grp_pop["Share_Pct"] = (grp_pop["Population"] / year_totals) * 100

# Order stack cleanly from bottom to top
group_order = ["Africa", "Asia", "Europe", "Rest of the world"]
grp_pop["Group"] = pd.Categorical(
    grp_pop["Group"], categories=group_order, ordered=True
)
grp_pop = grp_pop.sort_values(["Year", "Group"])

# 3. CVD-safe palette: Okabe-Ito blue accent for Africa + neutral greys for context
color_map = {
    "Africa": "#0072B2",
    "Asia": "#BDBDBD",
    "Europe": "#7F7F7F",
    "Rest of the world": "#E0E0E0",
}

# 4. Build 100% stacked bar chart
fig = px.bar(
    grp_pop,
    x="Year",
    y="Share_Pct",
    color="Group",
    category_orders={"Group": group_order},
    color_discrete_map=color_map,
    barmode="stack",
    template="plotly_white",
    title=(
        "<b>Africa's share of the world nearly doubled since 1970 while Europe's shrank</b>"
        "<br><span style='font-size:13px; color:#555555;'>"
        "Share of global population (%), 1970–2022; Africa highlighted in blue</span>"
    ),
)

# 5. Refine bars and hover information
fig.update_traces(
    marker_line_width=0.6,
    marker_line_color="#FFFFFF",
    hovertemplate=(
        "<b>%{fullData.name}</b> (%{x})<br>"
        "World share: <b>%{y:.1f}%</b><extra></extra>"
    ),
)

# 6. Direct annotations inside/beside key segments (1970 vs. 2022)
s_1970 = grp_pop[grp_pop["Year"] == 1970].set_index("Group")["Share_Pct"]
s_2022 = grp_pop[grp_pop["Year"] == 2022].set_index("Group")["Share_Pct"]

fig.add_annotation(
    x=1970,
    y=s_1970["Africa"] / 2,
    text=f"<b>{s_1970['Africa']:.1f}%</b>",
    showarrow=False,
    font=dict(size=11, color="#FFFFFF"),
)
fig.add_annotation(
    x=2022,
    y=s_2022["Africa"] / 2,
    text=f"<b>Africa {s_2022['Africa']:.1f}%</b>",
    showarrow=False,
    font=dict(size=11, color="#FFFFFF"),
)

eur_1970_mid = s_1970["Africa"] + s_1970["Asia"] + s_1970["Europe"] / 2
eur_2022_mid = s_2022["Africa"] + s_2022["Asia"] + s_2022["Europe"] / 2
fig.add_annotation(
    x=1970,
    y=eur_1970_mid,
    text=f"Eur {s_1970['Europe']:.1f}%",
    showarrow=False,
    font=dict(size=10, color="#FFFFFF"),
)
fig.add_annotation(
    x=2022,
    y=eur_2022_mid,
    text=f"Eur {s_2022['Europe']:.1f}%",
    showarrow=False,
    font=dict(size=10, color="#FFFFFF"),
)

asia_1970_mid = s_1970["Africa"] + s_1970["Asia"] / 2
asia_2022_mid = s_2022["Africa"] + s_2022["Asia"] / 2
fig.add_annotation(
    x=1970,
    y=asia_1970_mid,
    text=f"Asia {s_1970['Asia']:.1f}%",
    showarrow=False,
    font=dict(size=11, color="#333333"),
)
fig.add_annotation(
    x=2022,
    y=asia_2022_mid,
    text=f"Asia {s_2022['Asia']:.1f}%",
    showarrow=False,
    font=dict(size=11, color="#333333"),
)

fig.add_annotation(
    x=2022,
    y=95,
    text=f"Rest: {s_2022['Rest of the world']:.1f}%",
    showarrow=False,
    font=dict(size=10, color="#444444"),
)

# 7. Declutter layout and axes
fig.update_layout(
    showlegend=False,
    margin=dict(l=45, r=25, t=80, b=45),
    hovermode="x unified",
)

fig.update_xaxes(
    title=None,
    type="category",
    showgrid=False,
    linecolor="#CCCCCC",
)

fig.update_yaxes(
    title=None,
    showgrid=False,
    zeroline=False,
    ticksuffix="%",
    range=[0, 100],
)

fig.show()